# Notebook 10 -- Three-Angle + Operation-Type Evaluation on ASDiv
## LLM-to-SLM Guided Reasoning Pipeline (NVIDIA API Guide)

**What changed from Notebook 9?**

| Component | Notebook 9 (SLM-SLM) | Notebook 10 (LLM-SLM) |
|---|---|---|
| Guide model | Fine-tuned Llama 3B (local, LoRA) | Llama 3 70B via NVIDIA API |
| Guide output | Steps with exact numbers | **Verbal steps only** (no numbers/equations) |
| Solver model | Llama 1.5B (local, unchanged) | Llama 1.5B (local, unchanged) |
| VRAM needed | ~4.5 GB (both models) | ~1.4 GB (solver only) |
| Guide params_B | 3.0 | 70.0 |

**Why verbal-only plans?**

The guide now describes *what reasoning steps to take* in plain English,
without performing any arithmetic itself. This cleanly separates the roles:
- **Guide (LLM 70B)**: understands the problem structure, identifies quantities and operations
- **Solver (SLM 1.5B)**: executes all arithmetic from scratch using the verbal plan

**ASDiv exclusive advantage: Angle 4 (per operation type)**

ASDiv tags every problem with `solution_type` (Addition, Subtraction, Multiplication,
Division, Multi-step). This lets us ask: *does verbal LLM guidance help more on
harder operations like Division than on simple Addition?*

**Pipeline**
```
Question --> Llama 70B API (Guide) --> Verbal Plan --> Llama 1.5B (Solver) x5 --> Vote --> Answer
Baseline : Question ------------------------------------------> Llama 1.5B (Solver) x5 --> Vote
```


In [1]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
# !pip install -q openai          # <-- NEW: NVIDIA API client
print("Done.")


Done.


In [ ]:
# CELL 2 -- HuggingFace login (only needed for solver model download)
from huggingface_hub import login
login("")
print("HuggingFace login done")


HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm
from openai import OpenAI   # NVIDIA API client

OUTPUT_DIR = "/kaggle/working/asdiv_eval_llm_slm"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")


PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB
Output  : /kaggle/working/asdiv_eval_llm_slm


In [ ]:
# CELL 4 -- Configuration
CONFIG = {
    # Guide: NVIDIA-hosted LLM (no local GPU needed for guide)
    "nvidia_api_key"      : "",   # <-- paste your NVIDIA API key here
    "llm_guide_model"     : "meta/llama3-70b-instruct",

    # Solver: local SLM (unchanged from Notebook 9)
    "response_model"      : "meta-llama/Llama-3.2-1B-Instruct",

    # Dataset
    "dataset_name"        : "EleutherAI/asdiv",
    "dataset_split"       : "validation",
    "max_eval_samples"    : 300,   # validation has ~2300; increase for full run
    "random_seed"         : 42,    # SAME seed as Notebook 9 for fair comparison

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.7,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 350, 

    # Compute cost tracking
    "guide_params_B"      : 70.0,  # Llama 3 70B (cloud, not local)
    "solver_params_B"     : 1.0,   # Llama 3.2 1B (local)

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "angle4_file"         : f"{OUTPUT_DIR}/angle4_by_operation_type.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    if k == "nvidia_api_key":
        print(f"  {'nvidia_api_key':<24}: {'✅ set' if v else '❌ MISSING -- paste key above'}")
    else:
        print(f"  {k:<24}: {v}")


Config ready:
  nvidia_api_key          : ✅ set
  llm_guide_model         : meta/llama3-70b-instruct
  response_model          : meta-llama/Llama-3.2-1B-Instruct
  dataset_name            : EleutherAI/asdiv
  dataset_split           : validation
  max_eval_samples        : 300
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.7
  guide_temperature       : 0.1
  refiner_temperature     : 0.3
  max_new_tokens          : 350
  guide_params_B          : 70.0
  solver_params_B         : 1.0
  results_file            : /kaggle/working/asdiv_eval_llm_slm/results.jsonl
  report_file             : /kaggle/working/asdiv_eval_llm_slm/eval_report.json
  angle1_file             : /kaggle/working/asdiv_eval_llm_slm/angle1_compute_efficiency.json
  angle2_file             : /kaggle/working/asdiv_eval_llm_slm/angle2_vote_consistency.json
  angle3_file             : /kaggle/working/asdiv_eval_llm_slm/angle3_confidence_calibration.json
  angle4_file             :

In [5]:
# CELL 5 -- Load ASDiv dataset
# ASDiv fields: body, question, solution_type, answer, formula
# We combine body + question, extract numeric answer, and map
# solution_type to a broad 5-category label for Angle 4 analysis.

print("Loading ASDiv from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Features : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Split size: {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record:")
for k, v in ex.items():
    print(f"  {k}: {v}")


# Map the fine-grained solution_type into 5 broad categories
OP_MAP = {
    "addition"         : "Addition",
    "sum"              : "Addition",
    "subtraction"      : "Subtraction",
    "difference"       : "Subtraction",
    "comparison"       : "Subtraction",
    "multiplication"   : "Multiplication",
    "division"         : "Division",
    "common-division"  : "Division",
    "floor-division"   : "Division",
}

def broad_op(solution_type):
    """Map fine-grained solution_type to one of 5 broad categories."""
    st = str(solution_type).lower().strip()
    for key, val in OP_MAP.items():
        if key in st:
            return val
    return "Multi-step"


def normalise_asdiv(item):
    """Convert ASDiv record to {question, answer, op_type} pipeline format."""
    q = item["body"].strip().rstrip(".") + " " + item["question"].strip()

    # Answer may be "10 cookies" -- keep only the leading numeric part
    raw_ans = str(item["answer"]).strip().replace(",", "")
    m = re.match(r"(-?[\d\.]+)", raw_ans)
    ans_str = m.group(1) if m else raw_ans

    # Normalise float: 5.0 -> "5"
    try:
        f = float(ans_str)
        ans_str = str(int(f)) if f == int(f) else str(round(f, 4))
    except Exception:
        pass

    op_type       = broad_op(item.get("solution_type", ""))
    solution_type = str(item.get("solution_type", ""))

    return {
        "question"      : q,
        "answer"        : ans_str,
        "op_type"       : op_type,
        "solution_type" : solution_type,
    }


all_data = [normalise_asdiv(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Show operation type distribution before sampling
op_counts = Counter(d["op_type"] for d in all_data)
print(f"\nOperation type distribution ({len(all_data)} total):")
for op, cnt in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")

# CRITICAL: fix seed ONCE here, before any sampling
# Use the same seed as Notebook 9 (42) so the exact same questions are evaluated
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

sampled_op = Counter(d["op_type"] for d in test_data)
print(f"\nSampled operation distribution:")
for op, cnt in sorted(sampled_op.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")

print(f"\nFirst Q : {test_data[0]['question'][:80]}...")
print(f"First A : {test_data[0]['answer']}  |  op: {test_data[0]['op_type']}")
print("ASDiv loaded")


Loading ASDiv from HuggingFace...


README.md:   0%|          | 0.00/494 [00:00<?, ?B/s]

asdiv/validation-00000-of-00001.parquet:   0%|          | 0.00/267k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/2305 [00:00<?, ? examples/s]

Splits   : ['validation']
Features : ['body', 'question', 'solution_type', 'answer', 'formula']
Split size: 2305

Example record:
  body: Seven red apples and two green apples are in the basket.
  question: How many apples are in the basket?
  solution_type: Addition
  answer: 9 (apples)
  formula: 7+2=9

Operation type distribution (2305 total):
  Multi-step          : 723
  Subtraction         : 568
  Addition            : 446
  Division            : 308
  Multiplication      : 260

Sampled 300 questions (seed=42)

Sampled operation distribution:
  Multi-step          : 89
  Subtraction         : 67
  Addition            : 61
  Multiplication      : 43
  Division            : 40

First Q : He also has a section filled with short story booklets. If each booklet has 9 pa...
First A : 441  |  op: Multiplication
ASDiv loaded


In [6]:
# CELL 6 -- Answer extraction (same as Notebook 9)

def normalise_num(s):
    """Canonical numeric string. 5.0 -> '5', 3.14 -> '3.14'."""
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError:
        return s


def extract_gt_answer(answer_str):
    """ASDiv GT is already cleaned in normalise_asdiv -- just normalise."""
    return normalise_num(str(answer_str))


def extract_pred_answer(text):
    """
    Multi-pattern extractor. Returns empty string on failure.
    Never falls back to a random number from the text.

    Priority:
      1. #### N          -- standard format we request
      2. \\boxed{N}      -- LaTeX style
      3. 'the answer is' -- common phrasing
      4. '= N' at end of line
      5. **N** at end    -- bold markdown
      6. 'therefore N'   -- conclusion phrases
    """
    # 1
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    # 2
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    # 3
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    # 4
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    # 5
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    # 6
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""


# --- Self-test ---
_tests = [
    ("#### 42",             "42"),
    ("#### 3.5",            "3.5"),
    ("\\boxed{100}",        "100"),
    ("The answer is 7",     "7"),
    ("Total = 20",          "20"),
    ("**200**.",            "200"),
    ("Therefore, 13",       "13"),
    ("Some unrelated text", ""),
]
ok = True
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    status = "OK" if got == exp else "FAIL"
    if got != exp: ok = False
    print(f"  {status}  '{txt[:35]}' -> '{got}' (expected '{exp}')")
print("\nAll extractor tests passed" if ok else "\nEXTRACTOR HAS FAILURES -- fix before running eval")


  OK  '#### 42' -> '42' (expected '42')
  OK  '#### 3.5' -> '3.5' (expected '3.5')
  OK  '\boxed{100}' -> '100' (expected '100')
  OK  'The answer is 7' -> '7' (expected '7')
  OK  'Total = 20' -> '20' (expected '20')
  OK  '**200**.' -> '200' (expected '200')
  OK  'Therefore, 13' -> '13' (expected '13')
  OK  'Some unrelated text' -> '' (expected '')

All extractor tests passed


In [7]:
# CELL 7 -- Set up NVIDIA API client (replaces local 3B guide model)
#
# The guide is now Llama 3 70B served via NVIDIA's inference API.
# No GPU memory is used for the guide -- only the solver loads locally.
#
# Get your free API key at: https://build.nvidia.com

assert CONFIG["nvidia_api_key"], (
    "❌ nvidia_api_key is empty! Paste your key into CONFIG in Cell 4."
)

nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=CONFIG["nvidia_api_key"],
)

# Quick connectivity test
try:
    _test = nvidia_client.chat.completions.create(
        model=CONFIG["llm_guide_model"],
        messages=[{"role": "user", "content": "Reply with just: OK"}],
        max_tokens=5,
        temperature=0.0,
    )
    print(f"✅ NVIDIA API connected | model: {CONFIG['llm_guide_model']}")
    print(f"   Test response: {_test.choices[0].message.content.strip()}")
except Exception as e:
    print(f"❌ NVIDIA API connection failed: {e}")
    raise


✅ NVIDIA API connected | model: meta/llama3-70b-instruct
   Test response: OK


In [8]:
# CELL 8 -- Load solver model (Llama 1.5B, local)
# Only the solver loads onto GPU -- much lower VRAM vs Notebook 9.

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    dtype=torch.float16,
    device_map="auto",
).eval()

solver_vram = torch.cuda.memory_allocated() / 1e9
total_vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Solver VRAM  : {solver_vram:.2f} GB  (guide is on NVIDIA cloud, not here)")
print(f"Total GPU    : {total_vram:.1f} GB")
print(f"Headroom     : {total_vram - solver_vram:.1f} GB")
print("Memory OK -- only solver is local")


Loading solver: meta-llama/Llama-3.2-1B-Instruct


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Solver VRAM  : 1.26 GB  (guide is on NVIDIA cloud, not here)
Total GPU    : 15.6 GB
Headroom     : 14.4 GB
Memory OK -- only solver is local


In [9]:
# CELL 9 -- Prompts and generation functions
#
# KEY CHANGE: GUIDE_SYSTEM now asks for VERBAL steps only.
# The guide describes WHAT operation to do and WHY -- no numbers, no equations.
# The solver receives this verbal plan and does all the number work from scratch.
#
# Separation of roles:
#   Guide  --> language understanding, problem decomposition, operation identification
#   Solver --> arithmetic execution guided by the verbal plan
#
# NOTE: The ASDiv-specific GUIDE_SYSTEM also asks the guide to identify
# the operation type (single-step vs multi-step) to help structure the plan.

GUIDE_SYSTEM = (
    "You are a math problem analyst. Your ONLY job is to identify the logical "
    "reasoning steps needed to solve the problem.\n\n"
    "STRICT RULES:\n"
    "- Write steps in plain English. Do NOT perform any arithmetic.\n"
    "- Do NOT write any numbers, equations, or calculations in your steps.\n"
    "- Do NOT give the final answer or any intermediate numeric result.\n"
    "- Each step must say WHAT operation to do and WHY, using the names of "
    "  quantities, not their values.\n"
    "- Identify if this is a single-step or multi-step problem and structure "
    "  your plan accordingly. Maximum 3 steps.\n\n"
    "BAD  (has numbers and calculations):\n"
    "  Step 1: 49 x 9 = 441\n"
    "  Step 2: Answer is 441\n\n"
    "GOOD (verbal description only):\n"
    "  Step 1: Identify the two quantities: number of booklets and pages per booklet.\n"
    "  Step 2: Multiply the number of booklets by pages per booklet, because "
    "    reading all booklets means reading all pages in each one.\n"
    "  Step 3: The result of that multiplication is the total pages needed.\n\n"
    "Output only the numbered steps. Nothing else."
)

SOLVE_SYSTEM = (
    "You are a math problem solver.\n"
    "You will be given a problem and a verbal reasoning plan.\n"
    "Follow the plan exactly. Compute each step numerically.\n"
    "No markdown. No bullet points. No headers.\n"
    "Write plain arithmetic steps only.\n"
    "Your absolute last line must be: #### [number]\n"
    "NEVER write ### or ** in your response.\n\n"
    "Example:\n"
    "Booklets = 49. Pages per booklet = 9.\n"
    "Total pages = 49 x 9 = 441.\n"
    "#### 441"
)

BASELINE_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Read the problem carefully. Solve step by step, showing every calculation.\n"
    "Your FINAL line must be exactly: #### [number]"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts on this problem gave different answers.\n"
    "Ignore all previous attempts. Re-solve completely from scratch.\n"
    "Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)


# ---- Local solver call (same as Notebook 9) ----
def run_solver(messages, max_tokens, temperature):
    """Call the local SLM solver and return generated text."""
    prompt = resp_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = resp_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(resp_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = resp_model.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = resp_tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return resp_tok.decode(new_toks, skip_special_tokens=True).strip()


# ---- NVIDIA API guide call (NEW) ----
def generate_plan(question, max_retries=3):
    """
    Call Llama 70B via NVIDIA API to produce a VERBAL reasoning plan.
    The plan contains no numbers or calculations -- only natural language
    descriptions of what steps the solver should follow.
    """
    messages = [
        {"role": "system", "content": GUIDE_SYSTEM},
        {"role": "user",   "content": f"Problem: {question}"},
    ]
    for attempt in range(max_retries):
        try:
            completion = nvidia_client.chat.completions.create(
                model       = CONFIG["llm_guide_model"],
                messages    = messages,
                temperature = CONFIG["guide_temperature"],
                max_tokens  = 300,
                stream      = False,
            )
            return completion.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)   # exponential backoff
            else:
                print(f"  [guide API error after {max_retries} attempts]: {e}")
                return ("Step 1: Identify all quantities mentioned in the problem.\n"
                        "Step 2: Determine the correct arithmetic operation based on "
                        "what the question is asking.\n"
                        "Step 3: Apply the operation to compute the answer.")


# ---- Solver functions (local SLM, same interface as Notebook 9) ----
def generate_guided(question, plan):
    """
    Solver receives the verbal plan from the LLM guide and executes the math.
    The plan is presented as a reasoning guide -- solver fills in all numbers.
    """
    content = (
        f"Problem: {question}\n\n"
        f"Reasoning plan (follow each step and compute the numbers):\n{plan}\n\n"
        f"Now solve step by step with all arithmetic:"
    )
    return run_solver(
        [{"role": "system", "content": SOLVE_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    return run_solver(
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_refiner(question, candidates):
    # Refiner runs WITHOUT plan -- plan may have caused the tie
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem: {question}\n\n"
        f"Previous attempts gave: {cands}\n"
        "Ignore all previous attempts. Solve from scratch:"
    )
    return run_solver(
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )


print("Generation functions ready")
print("  generate_plan()     -> NVIDIA Llama 70B API (verbal steps, no math)")
print("  generate_guided()   -> local 1B solver with verbal plan")
print("  generate_baseline() -> local 1B solver no plan")
print("  generate_refiner()  -> local 1B solver tie-breaker (no plan)")


Generation functions ready
  generate_plan()     -> NVIDIA Llama 70B API (verbal steps, no math)
  generate_guided()   -> local 1B solver with verbal plan
  generate_baseline() -> local 1B solver no plan
  generate_refiner()  -> local 1B solver tie-breaker (no plan)


In [10]:
# CELL 10 -- Voting logic with richer metrics (same as Notebook 9)

def vote_and_decide(answers, question, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Filters empty responses before counting.
    Returns dict with all metrics needed for the four angles.
    """
    valid = [a for a in answers if a and a.strip()]
    if not valid:
        valid = answers  # fallback

    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        # Tie: refiner runs WITHOUT plan
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted           = total - new_top_c
        vote_counts      = new_counts

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "correct_votes"    : correct_votes,
        "total_votes"      : total,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }


print("Voting logic ready")
print("  majority         -> clear winner across votes")
print("  refiner_tiebreak -> tie broken by refiner (no plan)")
print("  coin_flip        -> still tied after refiner")


Voting logic ready
  majority         -> clear winner across votes
  refiner_tiebreak -> tie broken by refiner (no plan)
  coin_flip        -> still tied after refiner


In [11]:
# CELL 11 -- Single question test (verify LLM guide + SLM solver pipeline)

print("=" * 65)
print("SINGLE QUESTION TEST  (ASDiv, LLM-SLM)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
op   = item["op_type"]
print(f"Question : {q}")
print(f"GT Answer: {gt}  |  Operation type: {op}")

# Guided
print("\n[1] LLM guide (70B via NVIDIA API) generating VERBAL plan...")
plan = generate_plan(q)
print(f"Plan (verbal steps only, no numbers):\n{plan}")

print(f"\n[2] Guided solver votes ({CONFIG['n_votes']}x, local 1B)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")

# Baseline
print("\n[3] Baseline votes (no plan, local 1B)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


SINGLE QUESTION TEST  (ASDiv, LLM-SLM)
Question : He also has a section filled with short story booklets. If each booklet has 9 pages and there are 49 booklets in the short story section, how many pages will Jack need to go through if he plans to read them all?
GT Answer: 441  |  Operation type: Multiplication

[1] LLM guide (70B via NVIDIA API) generating VERBAL plan...
Plan (verbal steps only, no numbers):
1. Identify the two quantities: number of booklets and pages per booklet.
2. Multiply the number of booklets by pages per booklet, because reading all booklets means reading all pages in each one.
3. The result of that multiplication is the total pages needed.

[2] Guided solver votes (5x, local 1B)...
  Vote 1: '437'  |  raw[:80]: To find out how many pages Jack needs to read, we first identify the quantities:
  Vote 2: '441'  |  raw[:80]: To find out how many pages Jack needs to go through, we first identify the two q
  Vote 3: '435'  |  raw[:80]: To find out how many pages Jack 

In [12]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided   (70B LLM verbal plan via API + 1B solver x5)
#   Mode B: Baseline (1B solver x5, no plan)
#
# op_type is stored in every record so Angle 4 can group by operation.
#
# API rate limit tip: free NVIDIA tier = ~5 req/s.
# generate_plan() will auto-retry with exponential backoff on 429 errors.

print(f"Dual evaluation: {len(test_data)} ASDiv questions")
print(f"Guide  : {CONFIG['llm_guide_model']} via NVIDIA API (verbal plans)")
print(f"Solver : {CONFIG['response_model']} local")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="ASDiv Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])
    op_type   = item["op_type"]

    # ---- GUIDED (LLM verbal plan + SLM solver) ------------------
    try:
        plan        = generate_plan(question)   # 70B API, verbal steps
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "op_type"          : op_type,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "correct_votes"    : g_dec["correct_votes"],
            "total_votes"      : g_dec["total_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
            "vote_counts"      : g_dec["vote_counts"],
            "plan"             : plan,
        })
    except RuntimeError as e:
        all_results.append({
            "mode": "guided", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # ---- BASELINE (no plan) -------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "op_type"          : op_type,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "correct_votes"    : b_dec["correct_votes"],
            "total_votes"      : b_dec["total_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : None,
            "vote_counts"      : b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] Guided: {g_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

with open(CONFIG["results_file"], "w") as f:
    for r in all_results + base_results:
        f.write(json.dumps(r) + "\n")

g_c = sum(r["correct"] for r in all_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nEvaluation complete.")
print(f"  Guided (LLM-SLM)  : {g_c}/{len(all_results)} = {g_c/len(all_results)*100:.1f}%")
print(f"  Baseline (SLM)    : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"  Delta             : +{(g_c/len(all_results) - b_c/len(base_results))*100:.1f} percentage points")


Dual evaluation: 300 ASDiv questions
Guide  : meta/llama3-70b-instruct via NVIDIA API (verbal plans)
Solver : meta-llama/Llama-3.2-1B-Instruct local
Each question: 5 guided votes + 5 baseline votes
-----------------------------------------------------------------
Starting fresh


ASDiv Eval:   0%|          | 0/300 [00:00<?, ?it/s]

  [ 25] Guided: 60.0%  Baseline: 44.0%  (17.7 min)
  [ 50] Guided: 58.0%  Baseline: 40.0%  (37.7 min)
  [ 75] Guided: 54.7%  Baseline: 38.7%  (57.9 min)
  [100] Guided: 60.0%  Baseline: 40.0%  (77.3 min)
  [125] Guided: 60.0%  Baseline: 42.4%  (95.3 min)
  [150] Guided: 57.3%  Baseline: 41.3%  (113.9 min)
  [175] Guided: 56.0%  Baseline: 40.6%  (132.5 min)
  [200] Guided: 56.5%  Baseline: 40.0%  (150.9 min)
  [225] Guided: 58.2%  Baseline: 39.1%  (166.9 min)
  [250] Guided: 57.2%  Baseline: 38.4%  (186.0 min)
  [275] Guided: 57.5%  Baseline: 38.2%  (206.3 min)
  [300] Guided: 58.7%  Baseline: 37.3%  (228.4 min)

Evaluation complete.
  Guided (LLM-SLM)  : 176/300 = 58.7%
  Baseline (SLM)    : 112/300 = 37.3%
  Delta             : +21.3 percentage points


In [13]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
# =================================================================
# NOTE: The 70B guide runs on NVIDIA's cloud, so its param-passes
# are NOT local GPU compute. We track local and total separately.
#
#   Baseline : 1B solver x5                       =  5.0B local
#   Guided   : 70B guide (API) x1 + 1B solver x5  =  5.0B local + 70B cloud
#   Upper    : 70B model x5 (hypothetical ceiling) = 350.0B
#
# The local compute is identical for both modes -- any accuracy gain
# comes purely from the quality of the verbal plan.
# =================================================================

G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]

guided_local_compute   = S * N              # 5.0
baseline_local_compute = S * N              # 5.0
guided_total_compute   = (G * 1) + (S * N)  # 75.0 (including cloud guide)
upper_compute          = G * N              # 350.0

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

g_eff       = g_acc / guided_local_compute
b_eff       = b_acc / baseline_local_compute
savings_pct = (1 - guided_total_compute / upper_compute) * 100

g_wasted       = sum(r["wasted_votes"] for r in all_results)
b_wasted       = sum(r["wasted_votes"] for r in base_results)
total_possible = len(all_results) * N

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n": 0, "correct": 0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (ASDiv, LLM-SLM)")
print("=" * 65)
print(f"\n  {'Setup':<36} | {'Local Compute':>14} | {'Accuracy':>9} | {'Acc/B':>7}")
print(f"  {'-'*36}-+-{'-'*14}-+-{'-'*9}-+-{'-'*7}")
print(f"  {'Baseline (1B x ' + str(N) + ', no guide)':<36} | {baseline_local_compute:>12.1f}B  | {b_acc:>8.1f}% | {b_eff:>6.3f}")
print(f"  {'Guided  (70B API x1 + 1B x' + str(N) + ')':<36} | {guided_local_compute:>12.1f}B  | {g_acc:>8.1f}% | {g_eff:>6.3f}")
print(f"  {'Upper   (70B x ' + str(N) + ', hypothetical)':<36} | {upper_compute:>12.1f}B  | {'(ceiling)':>9} |")
print(f"\n  Note: Guide (70B) runs on NVIDIA cloud. Local compute is identical")
print(f"        for both modes. Accuracy gain is purely from verbal plan quality.")
print(f"\n  Accuracy gain over baseline   : +{g_acc - b_acc:.1f} percentage points")
print(f"  Total compute savings vs upper: {savings_pct:.0f}% cheaper (inc. cloud guide)")
print(f"  Wasted votes saved            : {b_wasted - g_wasted}  ({g_wasted} guided vs {b_wasted} baseline)")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")
print(f"\n  Strategy breakdown (guided):")
for s, v in sorted(strategy_stats.items(), key=lambda x: -x[1]["n"]):
    acc_s = v["correct"] / v["n"] * 100 if v["n"] else 0
    print(f"    {s:<22}: {v['n']:>4} questions  {acc_s:>6.1f}% accuracy")

angle1 = {
    "dataset"               : "ASDiv",
    "pipeline"              : "LLM-SLM (70B API guide + 1B local solver)",
    "n_questions"           : len(all_results),
    "guided_local_compute_B": guided_local_compute,
    "guided_total_compute_B": guided_total_compute,
    "baseline_compute_B"    : baseline_local_compute,
    "upper_compute_B"       : upper_compute,
    "guided_accuracy"       : round(g_acc, 2),
    "baseline_accuracy"     : round(b_acc, 2),
    "accuracy_gain"         : round(g_acc - b_acc, 2),
    "compute_savings_pct"   : round(savings_pct, 1),
    "guided_efficiency"     : round(g_eff, 4),
    "baseline_efficiency"   : round(b_eff, 4),
    "guided_wasted_votes"   : g_wasted,
    "baseline_wasted_votes" : b_wasted,
    "wasted_votes_saved"    : b_wasted - g_wasted,
    "refiner_triggered"     : ref_triggered,
    "refiner_correct"       : ref_correct,
    "strategy_breakdown"    : strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")


ANGLE 1 -- COMPUTE EFFICIENCY  (ASDiv, LLM-SLM)

  Setup                                |  Local Compute |  Accuracy |   Acc/B
  -------------------------------------+----------------+-----------+--------
  Baseline (1B x 5, no guide)          |          5.0B  |     37.3% |  7.467
  Guided  (70B API x1 + 1B x5)         |          5.0B  |     58.7% | 11.733
  Upper   (70B x 5, hypothetical)      |        350.0B  | (ceiling) |

  Note: Guide (70B) runs on NVIDIA cloud. Local compute is identical
        for both modes. Accuracy gain is purely from verbal plan quality.

  Accuracy gain over baseline   : +21.3 percentage points
  Total compute savings vs upper: 79% cheaper (inc. cloud guide)
  Wasted votes saved            : -63  (696 guided vs 633 baseline)
  Refiner: 72 triggered, 17 correct (23.6%)

  Strategy breakdown (guided):
    majority              :  228 questions    66.7% accuracy
    coin_flip             :   48 questions    20.8% accuracy
    refiner_tiebreak      :   24 ques

In [14]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

g_corr = [r["vote_consistency"] for r in all_results  if r["correct"]]
b_corr = [r["vote_consistency"] for r in base_results if r["correct"]]

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (ASDiv, LLM-SLM)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes per question):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Lift     : {lift:.2f}x  (guided produces {lift:.1f}x more correct votes per question)")
print(f"\n  Per-question comparison (same questions, both modes):")
print(f"    Guided beats baseline : {guided_wins} / {len(all_results)} questions")
print(f"    Baseline beats guided : {baseline_wins} / {len(all_results)} questions")
print(f"    Equal                 : {tied} / {len(all_results)} questions")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gv, bv = g_dist[bkt], b_dist[bkt]
    sign = "+" if gv - bv >= 0 else ""
    print(f"  {bkt:<22} | {gv:>8} | {bv:>8} | {sign+str(gv-bv):>6}")
if g_corr:
    print(f"\n  Correct-question consistency: Guided={np.mean(g_corr)*100:.1f}%  Baseline={np.mean(b_corr)*100:.1f}%")
    print("    (High consistency + correct = genuine reliable solving, not lucky vote)")

angle2 = {
    "dataset": "ASDiv", "pipeline": "LLM-SLM", "n_questions": len(all_results),
    "guided_mean_consistency": round(g_mean, 4), "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4), "guided_wins": guided_wins,
    "baseline_wins": baseline_wins, "tied": tied,
    "guided_distribution": g_dist, "baseline_distribution": b_dist,
    "guided_correct_q_consistency":   round(np.mean(g_corr), 4) if g_corr else 0,
    "baseline_correct_q_consistency": round(np.mean(b_corr), 4) if b_corr else 0,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")


ANGLE 2 -- VOTE CONSISTENCY  (ASDiv, LLM-SLM)

  Mean correct-vote ratio (out of 5 votes per question):
    Guided   : 41.8%  (2.09 votes correct on average)
    Baseline : 32.2%  (1.61 votes correct on average)
    Lift     : 1.30x  (guided produces 1.3x more correct votes per question)

  Per-question comparison (same questions, both modes):
    Guided beats baseline : 141 / 300 questions
    Baseline beats guided : 77 / 300 questions
    Equal                 : 82 / 300 questions

  Bucket                 |   Guided | Baseline |   Diff
  -----------------------+----------+----------+-------
  all_wrong  (0%)        |       63 |       93 |    -30
  low       (1-39%)      |       65 |       86 |    -21
  medium  (40-79%)       |      113 |       90 |    +23
  high   (80-100%)       |       59 |       31 |    +28

  Correct-question consistency: Guided=63.1%  Baseline=65.6%
    (High consistency + correct = genuine reliable solving, not lucky vote)

Saved -> /kaggle/working/asdiv_eval_

In [15]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)",    lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",     lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket": name, "count": n, "accuracy": round(acc, 4),
                           "expected": mid, "gap": round(gap, 4)})
    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"  {'ECE':<26}   {ece:.4f}")
    print(f"  High-conf: {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    return ece, calib_out, false_conf


print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (ASDiv, LLM-SLM)")
print("=" * 65)
print("Ideal: accuracy at each confidence level matches that level.")
print("False confidence: model agrees on wrong answer with full certainty.")

g_ece, g_calib, g_false = calibration_report(all_results,  "GUIDED (LLM verbal plan + SLM solver)")
b_ece, b_calib, b_false = calibration_report(base_results, "BASELINE (no plan)")

improve = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE Summary:")
print(f"    Guided   ECE : {g_ece:.4f}")
print(f"    Baseline ECE : {b_ece:.4f}")
print(f"    Improvement  : {improve:.1f}% better calibrated")
print(f"\n  False Confidence: Guided={g_false}  Baseline={b_false}  Reduction={b_false-g_false}")

angle3 = {
    "dataset": "ASDiv", "pipeline": "LLM-SLM", "n_questions": len(all_results),
    "guided_ece": round(g_ece, 4), "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(improve, 2),
    "guided_false_confidence": g_false, "baseline_false_confidence": b_false,
    "false_conf_reduction": b_false - g_false,
    "guided_calibration": g_calib, "baseline_calibration": b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")


ANGLE 3 -- CONFIDENCE CALIBRATION  (ASDiv, LLM-SLM)
Ideal: accuracy at each confidence level matches that level.
False confidence: model agrees on wrong answer with full certainty.

  [GUIDED (LLM verbal plan + SLM solver)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |    65 |     90.8% |       90% |  0.008 | Good
  High       (0.60-0.80)     |    87 |     64.4% |       70% |  0.056 | Good
  Medium     (0.40-0.60)     |    96 |     51.0% |       50% |  0.010 | Good
  Low        (<0.40)         |    52 |     23.1% |       25% |  0.019 | Good
  ECE                          0.0247
  High-conf: 65 questions  |  Accuracy: 90.8%  |  Confidently WRONG: 6

  [BASELINE (no plan)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        

In [16]:
# CELL 16 -- ANGLE 4: ACCURACY BY OPERATION TYPE  (ASDiv exclusive)
# =================================================================
# This is the analysis only ASDiv enables.
# We measure guided vs baseline accuracy and vote consistency
# separately for each operation category:
#   Addition / Subtraction / Multiplication / Division / Multi-step
#
# Key question: does verbal LLM guidance help more on harder
# operation types (Division, Multi-step) than simpler ones (Addition)?
# =================================================================

g_by_op = defaultdict(list)
b_by_op = defaultdict(list)
for r in all_results:
    g_by_op[r.get("op_type", "Unknown")].append(r)
for r in base_results:
    b_by_op[r.get("op_type", "Unknown")].append(r)

all_ops = sorted(set(list(g_by_op.keys()) + list(b_by_op.keys())))

print("=" * 72)
print("ANGLE 4 -- ACCURACY BY OPERATION TYPE  (ASDiv exclusive, LLM-SLM)")
print("=" * 72)
print(f"\n  {'Operation':<16} | {'N':>4} | {'Guided':>8} | {'Baseline':>9} | {'Gain':>6} | {'Consistency lift':>17}")
print(f"  {'-'*16}-+-{'-'*4}-+-{'-'*8}-+-{'-'*9}-+-{'-'*6}-+-{'-'*17}")

op_results = {}
for op in all_ops:
    g_items = g_by_op.get(op, [])
    b_items = b_by_op.get(op, [])
    n = len(g_items)
    if n == 0: continue

    g_acc  = sum(r["correct"] for r in g_items) / n * 100
    b_acc  = sum(r["correct"] for r in b_items) / max(len(b_items), 1) * 100
    gain   = g_acc - b_acc
    sign   = "+" if gain >= 0 else ""

    g_cons = np.mean([r["vote_consistency"] for r in g_items]) * 100
    b_cons = np.mean([r["vote_consistency"] for r in b_items]) * 100 if b_items else 0
    c_lift = g_cons / max(b_cons, 1e-6)

    print(f"  {op:<16} | {n:>4} | {g_acc:>7.1f}% | {b_acc:>8.1f}% | {sign+str(round(gain,1))+'%':>6} | {c_lift:>6.2f}x  ({g_cons:.1f}% vs {b_cons:.1f}%)")
    op_results[op] = {
        "n": n,
        "guided_acc": round(g_acc, 2), "baseline_acc": round(b_acc, 2),
        "gain": round(gain, 2),
        "guided_consistency": round(g_cons, 2), "baseline_consistency": round(b_cons, 2),
        "consistency_lift": round(c_lift, 4),
    }

# Summary: which operation type benefits most from verbal LLM guidance?
gains = sorted(op_results.items(), key=lambda x: -x[1]["gain"])
print(f"\n  Operation types ranked by guidance gain:")
for op, v in gains:
    sign = "+" if v["gain"] >= 0 else ""
    print(f"    {op:<16}: {sign}{v['gain']}%  (guided={v['guided_acc']}%  baseline={v['baseline_acc']}%)")

angle4 = {"dataset": "ASDiv", "pipeline": "LLM-SLM", "by_operation_type": op_results}
with open(CONFIG["angle4_file"], "w") as f:
    json.dump(angle4, f, indent=2)
print(f"\nSaved -> {CONFIG['angle4_file']}")


ANGLE 4 -- ACCURACY BY OPERATION TYPE  (ASDiv exclusive, LLM-SLM)

  Operation        |    N |   Guided |  Baseline |   Gain |  Consistency lift
  -----------------+------+----------+-----------+--------+------------------
  Addition         |   61 |    59.0% |     45.9% | +13.1% |   1.06x  (40.7% vs 38.5%)
  Division         |   40 |    65.0% |     40.0% | +25.0% |   1.25x  (42.3% vs 33.9%)
  Multi-step       |   89 |    40.4% |     27.0% | +13.5% |   1.27x  (32.4% vs 25.6%)
  Multiplication   |   43 |    74.4% |     41.9% | +32.6% |   1.37x  (51.8% vs 37.9%)
  Subtraction      |   67 |    68.7% |     38.8% | +29.9% |   1.59x  (48.6% vs 30.5%)

  Operation types ranked by guidance gain:
    Multiplication  : +32.56%  (guided=74.42%  baseline=41.86%)
    Subtraction     : +29.85%  (guided=68.66%  baseline=38.81%)
    Division        : +25.0%  (guided=65.0%  baseline=40.0%)
    Multi-step      : +13.48%  (guided=40.45%  baseline=26.97%)
    Addition        : +13.11%  (guided=59.02%  bas

In [17]:
# CELL 17 -- Full Paper Summary (all four angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)
with open(CONFIG["angle4_file"]) as f: a4 = json.load(f)

n = a1["n_questions"]

print("=" * 68)
print("  ASDiv EVALUATION -- PAPER SUMMARY TABLE (LLM-SLM)")
print("=" * 68)
print(f"  Dataset : ASDiv  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Guide   : Llama 70B via NVIDIA API (verbal plans, no arithmetic)")
print(f"  Solver  : Llama 1.5B local")
print()

rows = [
    ["Metric",                   "Baseline",      "Guided (LLM-SLM)", "Change"],
    ["Overall Accuracy",
     str(a1['baseline_accuracy']) + "%",
     str(a1['guided_accuracy']) + "%",
     "+" + str(round(a1['guided_accuracy'] - a1['baseline_accuracy'], 1)) + " pts"],
    ["Local Compute Cost",
     str(a1['baseline_compute_B']) + "B param-passes",
     str(a1['guided_local_compute_B']) + "B (+ 70B cloud guide)",
     "Same local; gain from verbal plan"],
    ["Wasted Votes",
     str(a1['baseline_wasted_votes']),
     str(a1['guided_wasted_votes']),
     str(a1['wasted_votes_saved']) + " fewer"],
    ["Vote Consistency",
     str(round(a2['baseline_mean_consistency'] * 100, 1)) + "%",
     str(round(a2['guided_mean_consistency'] * 100, 1)) + "%",
     str(round(a2['consistency_lift'], 2)) + "x lift"],
    ["High-Agreement Questions",
     str(a2['baseline_distribution']['high   (80-100%)']),
     str(a2['guided_distribution']['high   (80-100%)']),
     ""],
    ["Guided Wins Per-Question",
     "--",
     str(a2['guided_wins']) + " / " + str(n),
     ""],
    ["ECE (lower = better)",
     str(a3['baseline_ece']),
     str(a3['guided_ece']),
     str(a3['ece_improvement_pct']) + "% better"],
    ["False Confidence Count",
     str(a3['baseline_false_confidence']),
     str(a3['guided_false_confidence']),
     str(a3['false_conf_reduction']) + " fewer"],
]

col_w = [28, 22, 26, 30]
sep   = "-+-".join("-" * w for w in col_w)
for i, row in enumerate(rows):
    line = " | ".join(str(cell).ljust(col_w[j]) for j, cell in enumerate(row))
    print("  " + line)
    if i == 0:
        print("  " + sep)

if a1.get("refiner_triggered", 0) > 0:
    rt = a1["refiner_triggered"]
    rc = a1.get("refiner_correct", 0)
    print(f"\n  Refiner: triggered {rt} times, resolved {rc} correctly ({rc/rt*100:.1f}%)")

# Angle 4 summary
print(f"\n  Accuracy gain by operation type (Angle 4):")
gains = sorted(a4["by_operation_type"].items(), key=lambda x: -x[1]["gain"])
for op, v in gains:
    sign = "+" if v["gain"] >= 0 else ""
    print(f"    {op:<16}: {sign}{v['gain']}%  "
          f"(guided={v['guided_acc']}%  baseline={v['baseline_acc']}%  "
          f"n={v['n']})")

full = {
    "dataset": "ASDiv", "seed": CONFIG["random_seed"],
    "pipeline": "LLM-SLM (70B verbal guide + 1B local solver)",
    "n_questions": n, "angle1": a1, "angle2": a2, "angle3": a3, "angle4": a4,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(full, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}")
print("Commit this notebook to preserve outputs.")


  ASDiv EVALUATION -- PAPER SUMMARY TABLE (LLM-SLM)
  Dataset : ASDiv  |  N=300  |  Seed=42
  Guide   : Llama 70B via NVIDIA API (verbal plans, no arithmetic)
  Solver  : Llama 1.5B local

  Metric                       | Baseline               | Guided (LLM-SLM)           | Change                        
  -----------------------------+------------------------+----------------------------+-------------------------------
  Overall Accuracy             | 37.33%                 | 58.67%                     | +21.3 pts                     
  Local Compute Cost           | 5.0B param-passes      | 5.0B (+ 70B cloud guide)   | Same local; gain from verbal plan
  Wasted Votes                 | 633                    | 696                        | -63 fewer                     
  Vote Consistency             | 32.2%                  | 41.8%                      | 1.3x lift                     
  High-Agreement Questions     | 31                     | 59                         |              